In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'ar5'

def decay(t0,
          t,
          alpha,
          tau):
    if type(alpha) == float:
        return np.exp(-(t-t0)/tau)
    elif len(alpha) > 1: # probably CO2
        assert len(alpha) == len(tau) + 1
        return alpha[0] + sum(aa * np.exp(-(t-t0)/tt) for aa,tt in zip(alpha[1:],tau))

def concentration(emission_profile, # list of coordinates (t,y), t is strictly increasing
                  tau,
                  alpha=1.0,
                  step=1,
                  c_0=0.0,
                  t_max=2100):
    
    '''
    
    
    
    '''
    
    if type(alpha) == float:
        CO2 = False
    else:
        CO2 = True
    
    n_points = len(emission_profile)
    assert n_points >= 2
    
    # initialize
    T0 = emission_profile[0][0]
    c = np.zeros(shape=t_max-T0+1)
    
    for i in range(n_points-1):
        
        # extract the variables that we need
        t0, y0 = emission_profile[i]
        t1, y1 = emission_profile[i+1]
        
        # create the time series for the interval of interest
        time = np.arange(t0,t1+1)
        
        # calculate the linear coefficients
        a = (y1-y0)/(t1-t0)
        b = (y0*t1-y1*t0)/(t1-t0)
        
        print(t0,t1,y0,y1,a,b)
        
        
        if CO2:
            # calculate the decay function
            d = decay(t0,time,alpha,tau) # this should be fine (integrated until t_max)
            
            c_i = list(alpha[0]*(a*(time**2-t0**2)/2+b*(time-t0)) + \
                sum(aa*tt*(a*((tt-t0)*np.exp(-(time-t0)/tt)+time-tt) + b*(1-np.exp(-(time-t0)/tt))) for aa,tt in zip(alpha[1:],tau)))
            
        else:
            # calculate the decay function
            d = np.exp(-(time-t0)/tau)
        
            # calculate the dynamic concentrations
            c_i = list(c_0*d + # any residue from a previous period \
                tau*(a*((tau-t0)*d+time-tau) + b*(1-d))) # plus the ongoing emissions
        
#    if t1 < t_max:
        c_0 = c_i.pop()
        time_to_end = np.arange(t1,t_max+1)
        c_i.extend(c_0*decay(t1,time_to_end,alpha,tau))
        # finally pop the last value, that's the start of the next period
        
        # store it
        c += np.array([0.] * (t0-T0) + c_i)
    
    # if we want to calculate the concentration beyond the last t, then let's do that
        print(c.shape)
    
    emissions = list(zip(*emission_profile))
    
    time = np.arange(T0,t_max+1)
    concentration_series = pd.Series(c, index=time)
    
    emission_series = pd.Series(emissions[1],
                                index=emissions[0]).reindex(time).interpolate(limit_area='inside')
    
    return emission_series, concentration_series


In [ ]:
a_x_i = pd.read_csv(DATA_DIR / 'ghg_properties.csv', index_col=0)
a_x_i['Lifetime (Years)'] = pd.to_numeric(a_x_i['Lifetime (Years)'],errors='coerce')
a_x_i

In [ ]:
a_x_i['Lifetime (Years)']['Methane']

In [ ]:
atm_mass   = 5.1480e18 # kg
atm_mol    = 29.80 # g/mol

# CO2
alpha_CO2 = [0.217278,0.224037,0.282381,0.276303]
tau_CO2   = [394.409,36.5393,4.30365]
rf_ppb_CO2 = a_x_i['Radiative Efficiency (W m-2 ppb-1)']['Carbon dioxide']
MM_CO2 = a_x_i['Molar mass (g mol-1)']['Carbon dioxide']

# CH4
alpha_CH4 = 1.
tau_CH4   = a_x_i['Lifetime (Years)']['Methane']
rf_ppb_CH4 = a_x_i['Radiative Efficiency (W m-2 ppb-1)']['Methane']
MM_CH4 = a_x_i['Molar mass (g mol-1)']['Methane']

# profile in kg
profile = ((2000, 20),
           (2010, 30),
           (2020, 35),
           (2030, 38),
           (2040, 40),
           (2050, 40))

profile = ((2000, 20),
           (2010, 30),
           (2020, 40))
#
#profile = ((2000, 20),
#           (2020, 40))

#profile = ((2000, 50),
#           (2050, 30))

profile_CO2 = ((2000, 40e12), # in Gt
          (2200, 40e12))
profile_CH4 = ((2000, .4e12), # in Gt
          (2200, .4e12))

a_CO2,b_CO2   = concentration(profile_CO2,
                      alpha=alpha_CO2,
                      tau=tau_CO2,
                      c_0=275,
                             t_max=2200)

a_CH4,b_CH4   = concentration(profile_CH4,
                      alpha=alpha_CH4,
                      tau=tau_CH4,
                      c_0=275,
                             t_max=2200)

results = pd.concat([a_CO2,
                     a_CH4,
          rf_ppb_CO2/MM_CO2 * atm_mol/atm_mass * 1e9 * b_CO2,
          
          rf_ppb_CH4/MM_CH4 * atm_mol/atm_mass * 1e9 * b_CH4],
          keys = pd.MultiIndex.from_product([['Emissions','Radiative forcing'],['CO2','CH4']]),
    axis=1)


In [ ]:
results

In [ ]:
results['Emissions'].plot(title='Emissions (kg/year)',logy=True)
results['Radiative forcing'].plot(title='Warming (W m-2)')